# Caso Bookshop – Limpieza y Análisis de Ventas

Este notebook resuelve el caso *Bookshop*, donde se trabaja con dos fuentes de datos:

- Ventas de **tiendas físicas** (`bookshop-stores-sales.txt`)
- Ventas del **sitio web** (`bookshop-web-sales.txt`)

**Objetivos:**

1. Unificar la información de ventas de tiendas y web.
2. Construir un dataset final con las columnas:
   - Store  
   - Title  
   - Units sold  
   - List price  
   - Royalty  
3. Exportar el resultado a **Excel** para la gerencia.
4. Responder:
   - ¿Cuáles son los libros más vendidos?
   - ¿Cuáles son los libros que más conviene vender (mayor ingreso total)?


In [ ]:
# ===============================================
# 1. Importar librerías
# ===============================================
# pandas: manejo de datos en formato tabular
# numpy: operaciones numéricas
import pandas as pd
import numpy as np


In [ ]:
# ===============================================
# 2. Cargar archivos de ventas
# ===============================================
# Cargamos los archivos de texto entregados en el caso:
# - bookshop-stores-sales.txt : ventas en tiendas físicas
# - bookshop-web-sales.txt    : ventas realizadas a través del sitio web

df_store = pd.read_csv('bookshop-stores-sales.txt')
df_web = pd.read_csv('bookshop-web-sales.txt')

print("Información df_store (tiendas físicas):")
df_store.info()
print("\nInformación df_web (ventas web):")
df_web.info()


In [ ]:
# Vista rápida de las primeras filas de cada fuente
df_store.head(10)


In [ ]:
df_web.head(10)


## 3. Homologar nombres de columnas

El objetivo es que ambos DataFrames (store y web) tengan los mismos nombres de columnas, en español y alineados con los nombres solicitados en el enunciado.


In [ ]:
# --- Ventas web ---
df_web.rename(
    columns={
        'Book title': 'Título',
        'Number sold': 'Cantidad vendida',
        'Sales price': 'Precio de venta',
        'Royalty paid': 'Royalty'
    },
    inplace=True
)

df_web.head(10)


In [ ]:
# --- Ventas en tiendas físicas ---
df_store.rename(
    columns={
        'Title': 'Título',
        'Units sold': 'Cantidad vendida',
        'List price': 'Precio de venta',
        'Royalty paid': 'Royalty'
    },
    inplace=True
)

df_store.head(10)


## 4. Limpieza de datos de tiendas físicas

En el archivo de tiendas físicas (`df_store`) vienen filas que no corresponden directamente a títulos de libros (por ejemplo, totales o filas de separación). En esta sección se:
- Elimina filas completamente vacías.
- Separa las filas correspondientes a **tiendas de EE.UU. (US)** y **Europa (EUR)**.
- Se crea una columna `Store` para identificar el origen de la venta.


In [ ]:
# Eliminar filas que están completamente vacías
df_store.dropna(how='all', inplace=True)
df_store


In [ ]:
# -----------------------------------------------
# 4.1. Extraer filas correspondientes a US
# -----------------------------------------------
# Según la estructura del archivo, las primeras filas corresponden a las tiendas de US.
df_storeUS = df_store.iloc[0:8].copy()
df_storeUS.reset_index(drop=True, inplace=True)
df_storeUS


In [ ]:
# Eliminar filas que no representan títulos de libros
# (por ejemplo, cabeceras o totales dentro del archivo original).
df_storeUS = df_storeUS.drop([0, 1])
df_storeUS.reset_index(drop=True, inplace=True)

# Agregar columna que identifique el origen de la venta
df_storeUS['Store'] = 'US'
df_storeUS


In [ ]:
# -----------------------------------------------
# 4.2. Extraer filas correspondientes a EUR
# -----------------------------------------------
# A partir de la fila 8 en adelante tenemos la información de Europa.
df_storeEUR = df_store.iloc[8::].copy()
df_storeEUR


In [ ]:
# Eliminar filas que no son títulos (cabeceras / totales)
df_storeEUR = df_storeEUR.drop([11, 12])
df_storeEUR.reset_index(drop=True, inplace=True)
df_storeEUR


In [ ]:
# Eliminar otras filas que no son títulos (según la revisión manual)
df_storeEUR = df_storeEUR.drop([4, 5])
df_storeEUR.reset_index(drop=True, inplace=True)
df_storeEUR


In [ ]:
# Agregar columna que identifique el origen de la venta
df_storeEUR['Store'] = 'EUR'
df_storeEUR


In [ ]:
# -----------------------------------------------
# 4.3. Unificar tiendas físicas (US + EUR)
# -----------------------------------------------
df_Storenew = pd.concat([df_storeUS, df_storeEUR], ignore_index=True)
df_Storenew


## 5. Preparar datos de ventas web


In [ ]:
# Creamos también la columna 'Store' para el canal web,
# de manera de poder unificar todas las ventas.
df_web['Store'] = 'Web'
df_web.head(10)


## 6. Unificación de tiendas físicas y web


In [ ]:
# Concatenamos los DataFrames:
# - df_Storenew : tiendas US y EUR
# - df_web      : ventas web
df_final = pd.concat([df_Storenew, df_web], ignore_index=True)
df_final


In [ ]:
# Eliminar filas donde 'Cantidad vendida' es NaN,
# ya que no aportan información al análisis.
df_final = df_final.dropna(subset=['Cantidad vendida'])
df_final


In [ ]:
# Reordenar columnas para que queden tal como se solicita:
# Store, Título, Cantidad vendida, Precio de venta, Royalty
df_final = df_final[['Store', 'Título', 'Cantidad vendida', 'Precio de venta', 'Royalty']]
df_final


In [ ]:
# Ordenar por Store (solo por estética del reporte final)
df_final = df_final.sort_values(by='Store', ascending=False)
df_final


In [ ]:
# Renombrar columnas a los nombres definitivos en inglés,
# tal como se pide en el enunciado.
df_final.rename(
    columns={
        'Store': 'Store',
        'Título': 'Title',
        'Cantidad vendida': 'Units sold',
        'Precio de venta': 'List price',
        'Royalty': 'Royalty'
    },
    inplace=True
)

df_final.head()


## 7. Exportar reporte final a Excel


In [ ]:
# Guardamos el DataFrame consolidado en un archivo Excel
# que puede ser utilizado por la gerencia.
df_final.to_excel('bookshop-sales-data.xlsx', index=False)


## 8. Análisis de resultados

A partir del dataset consolidado `df_final`, se responden las preguntas:

1. ¿Cuáles son los libros más vendidos (en unidades)?
2. ¿Cuáles son los libros que generan mayor ingreso total (unidades × precio).


In [ ]:
# -----------------------------------------------
# 8.1. ¿Cuáles son los libros más vendidos?
# -----------------------------------------------
# Agrupamos por título y sumamos las unidades vendidas.
mas_vendidos = (
    df_final
    .groupby('Title', as_index=False)['Units sold']
    .sum()
    .sort_values(by='Units sold', ascending=False)
)

print("Libros ordenados por unidades vendidas (de mayor a menor):")
print(mas_vendidos)

print(f"\nEl libro más vendido es: {mas_vendidos.iloc[0, 0]}")


In [ ]:
# -----------------------------------------------
# 8.2. ¿Cuáles son los libros que más conviene vender?
# -----------------------------------------------
# Creamos una columna de ventas totales en dinero:
# Total Sales = Units sold * List price
df_final['Total Sales'] = df_final['Units sold'] * df_final['List price']

# Agrupamos por título y sumamos el monto total de ventas
mas_convenientes = (
    df_final
    .groupby('Title', as_index=False)['Total Sales']
    .sum()
    .sort_values(by='Total Sales', ascending=False)
)

print("Libros ordenados por ingreso total (de mayor a menor):")
print(mas_convenientes)


## 9. Conclusiones

- Se integraron correctamente las fuentes de datos de tiendas físicas (US/EUR) y ventas web.
- Se generó un archivo Excel (`bookshop-sales-data.xlsx`) con el formato solicitado para la gerencia.
- A partir del análisis, se identificaron los libros con mayor cantidad de unidades vendidas y aquellos que generan mayor ingreso total.
